# GEP Z-Region Experiment ($\mathcal{Z}_n^1$ vs $\mathcal{Z}_n^2$)

This notebook runs 6 settings for GEP:
- DBC + $\mathcal{Z}_n^1$ (copy variable continuous in [0,1])
- DBC + $\mathcal{Z}_n^2$ (copy variable binary)
- LC  + $\mathcal{Z}_n^1$
- LC  + $\mathcal{Z}_n^2$
- SMC + $\mathcal{Z}_n^1$
- SMC + $\mathcal{Z}_n^2$


In [ ]:
using Pkg, Dates

function find_repo_root(start::AbstractString = pwd())
    dir = abspath(start)
    while true
        if isfile(joinpath(dir, "Project.toml"))
            return dir
        end
        parent = dirname(dir)
        parent == dir && error("Project.toml not found")
        dir = parent
    end
end

repo_root = find_repo_root()
cd(repo_root)
Pkg.activate(repo_root)

println("Repo root: ", repo_root)
println("Timestamp: ", Dates.format(now(), "yyyy-mm-dd HH:MM:SS"))


In [ ]:
configs = [
    "experiments/generation_expansion/configs/z_region/ge_dbc_z1.jl",
    "experiments/generation_expansion/configs/z_region/ge_dbc_z2.jl",
    "experiments/generation_expansion/configs/z_region/ge_lc_z1.jl",
    "experiments/generation_expansion/configs/z_region/ge_lc_z2.jl",
    "experiments/generation_expansion/configs/z_region/ge_smc_z1.jl",
    "experiments/generation_expansion/configs/z_region/ge_smc_z2.jl",
]

for cfg in configs
    println("Running: ", cfg)
    run(`julia --project=. experiments/generation_expansion/run_experiment.jl $cfg`)
end


In [ ]:
using JLD2, DataFrames, Statistics, Printf

function parse_num(v)
    if v isa Missing
        return NaN
    elseif v isa AbstractString
        x = tryparse(Float64, replace(strip(v), "%" => ""))
        return isnothing(x) ? NaN : x
    else
        return Float64(v)
    end
end

vecnum(v) = [parse_num(x) for x in v]

function summary_for_tag(root::AbstractString, tag::AbstractString)
    files = String[]
    for (r, _, fs) in walkdir(root)
        for f in fs
            if endswith(f, ".jld2") && occursin("tag=" * tag, f)
                push!(files, joinpath(r, f))
            end
        end
    end
    sort!(files)

    rows = NamedTuple[]
    for p in files
        d = load(p)
        sh = d["sddp_results"][:sol_history]
        namestr = String.(names(sh))
        lb = vecnum(sh[!, ("LB" in namestr ? "LB" : "lb")])
        ub = vecnum(sh[!, ("UB" in namestr ? "UB" : "ub")])
        t = vecnum(sh[!, ("Time" in namestr ? "Time" : "time")])

        best_lb = copy(lb)
        for i in 2:length(best_lb)
            best_lb[i] = max(best_lb[i], best_lb[i-1])
        end
        best_ub = copy(ub)
        for i in 2:length(best_ub)
            best_ub[i] = min(best_ub[i], best_ub[i-1])
        end

        _, idx = findmax(best_lb)
        time_lb = t[idx]
        gap = 100.0 * (best_ub[end] - best_lb[end]) / abs(best_ub[end])
        solved = gap <= 0.1

        push!(rows, (file=basename(p), gap=gap, time_lb=time_lb, solved=solved))
    end

    solved_n = count(r->r.solved, rows)
    gap_mean = mean([r.gap for r in rows])
    tlb_mean = mean([r.time_lb for r in rows])
    return rows, (solved=solved_n, total=length(rows), gap=gap_mean, time=tlb_mean)
end

root = joinpath("results", "ge_z_region_runs", "generation_expansion")
for (label, tag) in [
    ("DBC+Z1", "ge_zregion_dbc_z1"),
    ("DBC+Z2", "ge_zregion_dbc_z2"),
    ("LC+Z1",  "ge_zregion_lc_z1"),
    ("LC+Z2",  "ge_zregion_lc_z2"),
    ("SMC+Z1", "ge_zregion_smc_z1"),
    ("SMC+Z2", "ge_zregion_smc_z2"),
]
    _, s = summary_for_tag(root, tag)
    @printf("%-8s  solved=%d/%d  gap=%.2f%%  timeLB=%.1fs
", label, s.solved, s.total, s.gap, s.time)
end
